# পাঠ ০৮ - মাল্টি-এজেন্ট ডিজাইন প্যাটার্ন


## সেটআপ


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## কেন মাল্টি-এজেন্ট সিস্টেম?

ট্রিপ পরিকল্পনার মতো বাস্তব-জীবনের কাজগুলিতে বিভিন্ন ধরনের দক্ষতার প্রয়োজন হয় — লজিস্টিকস, স্থানীয় জ্ঞান, বাজেটিং, এবং আরও অনেক কিছু। সবকিছু একক এজেন্ট দ্বারা পরিচালনা করা দ্রুত জটিল হয়ে যায়।

মাল্টি-এজেন্ট সিস্টেমগুলি এটি **বিশেষজ্ঞতা**র মাধ্যমে সমাধান করে: প্রতিটি এজেন্ট একটি দক্ষতার ক্ষেত্রে ফোকাস করে, যা একটি সাধারণতাবাদের চেয়ে উচ্চমানের ফলাফল উৎপন্ন করে। এছাড়াও এগুলি **স্কেলেবিলিটি** উন্নত করে — আপনি নতুন এজেন্ট যুক্ত করতে পারেন (যেমন, একটি ফ্লাইট স্পেশালিস্ট, একটি রেস্টুরেন্ট সমালোচক) বিদ্যমান ওয়ার্কফ্লো পুনর্লিখন ছাড়াই। এজেন্টগুলি একটি কাঠামোবদ্ধ পাইপলাইনের মাধ্যমে একসাথে যুক্ত হয়, একটির থেকে অন্যটিতে প্রসঙ্গ প্রেরণ করে।


## বিশেষজ্ঞ এজেন্ট তৈরি করা


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## একটি ধারাবাহিক ওয়ার্কফ্লো তৈরি করা

`WorkflowBuilder` আপনাকে এজেন্টদের একটি নির্দেশিত গ্রাফে সংযুক্ত করার সুযোগ দেয়। এখানে আমরা একটি সহজ দুটি ধাপের পাইপলাইন তৈরি করছি: **TravelPlanner** যাত্রা সূচি প্রস্তুত করে, তারপর **TravelConcierge** সেটি পর্যালোচনা করে এবং উন্নত করে।


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ওয়ার্কফ্লোতে আরও এজেন্ট যোগ করা

মাল্টি-এজেন্ট প্যাটার্নের সবচেয়ে বড় সুবিধাগুলির একটি হল এটি কত সহজে সম্প্রসারিত করা যায়। নিচে আমরা একটি **BudgetReviewer** এজেন্ট যোগ করেছি যা ভ্রমণকারীর বাজেটের বিরুদ্ধে পরিকল্পনাটি পরীক্ষা করে, এমন আইটেমগুলো চিহ্নিত করে যেগুলো খরচ সীমা ছাড়িয়ে যেতে পারে এবং টাকা সাশ্রয়ের বিকল্প প্রস্তাব করে। ওয়ার্কফ্লো এখন তিনটি এজেন্টকে ক্রমানুসারে চালায়:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## সারসংক্ষেপ

এই পাঠে আপনি শিখেছেন কীভাবে:

1. **বিশেষায়িত এজেন্ট তৈরি করবেন** — প্রত্যেকটি একটি মনোনিবেশিত ভূমিকা সহ (পরিকল্পনা, কনসিয়ার্জ, বাজেট পর্যালোচনা)।
2. **`WorkflowBuilder` এবং `add_edge` ব্যবহার করে এজেন্টদের ধারাবাহিক ওয়ার্কফ্লোতে সংযুক্ত করবেন**।
3. **মাল্টি-এজেন্ট পাইপলাইনের আউটপুট স্ট্রিম করবেন, ট্র্যাক করবেন কে কোন এজেন্ট বলছে**।
4. **কোনও বিদ্যমান এজেন্ট পরিবর্তন না করে নতুন এজেন্ট যোগ করে ওয়ার্কফ্লো প্রসারিত করবেন**।

মাল্টি-এজেন্ট ডিজাইন প্যাটার্ন প্রত্যেক এজেন্টকে সহজ রাখে, তবুও একক এজেন্টের তুলনায় আরো সমৃদ্ধ এবং বিস্তারিতভাবে পর্যালোচিত ফলাফল উৎপন্ন করে।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
